# 11. Interval catalogue

All intervals of notebooks 03–08 in one table: `results/intervals_catalogue.csv`

- `unit`: resampling unit (seed, file, site, logger, physical run)
- `kind`: level or difference between protocols
- `n_units`: number of independent units
- Seed-level intervals: split/fitting variability only

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import src as la
from src import config as cfg, sfb, evaluation as ev, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
R = cfg.RESULTS
cat = []

def add(exp, quantity, est, lo, hi, unit, n_units, kind="level", note=""):
    cat.append(dict(experiment=exp, quantity=quantity, estimate=est, lo=lo, hi=hi,
                    unit=unit, n_units=n_units, kind=kind, note=note))

def seed_ci(df, keys, exp, value="acc"):
    for key, g in df.groupby(keys):
        ci = sfb.bootstrap_ci(g[value].to_numpy(), n_boot=cfg.N_BOOT)
        add(exp, " | ".join(map(str, key if isinstance(key, tuple) else (key,))) + f" | {value}",
            ci["mean"], ci["lo"], ci["hi"], "seed (split + initialisation)", ci["n"],
            note="fitting/split variability, not sampling variability")

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. D1 (E4, E4b, E10)

In [2]:
if (R / "E4_collapse.csv").exists():
    d4 = pd.read_csv(R / "E4_collapse.csv")
    seed_ci(d4, ["task", "model", "protocol"], "E4 D1 collapse")
    t4 = pd.DataFrame([c for c in cat if c["experiment"] == "E4 D1 collapse"])
    t4.to_csv(R / "E4_collapse_ci.csv", index=False)
for f, exp in [("E4_site_cluster_ci_seed0.csv", "E4 D1 collapse (seed 0)")]:
    if (R / f).exists():
        for _, r in pd.read_csv(R / f).iterrows():
            add(exp, f"{r.task} | {r.protocol} | acc", r.acc, r.lo, r.hi, "site", r.n_test_sites)
if (R / "E4b_paired_oof.csv").exists():
    for _, r in pd.read_csv(R / "E4b_paired_oof.csv").dropna(subset=["d_acc"]).iterrows():
        for m in ["acc", "bacc", "excess"]:
            add("E4b D1 paired OOF", f"{r.task} | {r.comparison} | drop in {m}", r[f"d_{m}"],
                r[f"d_{m}_lo"], r[f"d_{m}_hi"], "site", r.n_clusters, kind="difference")
if (R / "E10_mitigation_raw.csv").exists():
    seed_ci(pd.read_csv(R / "E10_mitigation_raw.csv"), ["method"], "E10 D1 mitigation")
if (R / "E10_mitigation_diff_vs_ERM.csv").exists():
    for _, r in pd.read_csv(R / "E10_mitigation_diff_vs_ERM.csv").iterrows():
        add("E10 D1 mitigation", f"{r.method} - ERM | {r.metric}", r.diff_vs_ERM, r.lo, r.hi,
            "seed (paired split)", r.n_seeds, kind="difference")

## 2. D2 (E6, E7)

In [3]:
if (R / "E6_time_confound_D2.csv").exists():
    for _, r in pd.read_csv(R / "E6_time_confound_D2.csv").iterrows():
        add("E6 D2 clock stump", f"{r.population} | {r.protocol} | acc", r.acc, r.wilson_lo, r.wilson_hi,
            "file (Wilson, files independent)", r.n)
        add("E6 D2 clock stump", f"{r.population} | {r.protocol} | acc", r.acc, r.logger_boot_lo,
            r.logger_boot_hi, "logger", r.n_loggers)
        add("E6 D2 clock stump", f"{r.population} | {r.protocol} | excess over majority", r.excess,
            r.excess_lo, r.excess_hi, "logger", r.n_loggers)
if (R / "E7_foundation_raw.csv").exists():
    e7 = pd.read_csv(R / "E7_foundation_raw.csv")
    seed_ci(e7, ["population", "model", "protocol"], "E7 D2 pretrained (v1 design)")
    out = []
    for (pop, m), g in e7.groupby(["population", "model"]):
        a = g[g.protocol == "P1_naive"].sort_values("seed").acc.to_numpy()
        b = g[g.protocol == "P2_logger_indep"].sort_values("seed").acc.to_numpy()
        if len(a) and len(b):
            r = sfb.bootstrap_paired_diff(a, b, n_boot=cfg.N_BOOT)
            add("E7 D2 pretrained (v1 design)", f"{pop} | {m} | naive - logger-indep", r["diff"], r["lo"],
                r["hi"], "seed (different splits per protocol)", r["n"], kind="difference",
                note="splits of the two protocols differ; not a paired-unit test")
            out.append(dict(population=pop, model=m, collapse=r["diff"], ci_lo=r["lo"], ci_hi=r["hi"],
                            significant=r["excludes_zero"], unit="seed"))
    pd.DataFrame(out).to_csv(R / "E7_collapse_ci.csv", index=False)
if (R / "E7_acoustic_vs_clock.csv").exists():
    for _, r in pd.read_csv(R / "E7_acoustic_vs_clock.csv").iterrows():
        add("E7 D2 acoustic vs clock", f"{r.population} | {r.predictor} | acc", r.acc, r.lo_seed0, r.hi_seed0,
            "logger", r.n_loggers, note="fold seed 0")
if (R / "E7_acoustic_vs_clock_differences.csv").exists():
    for _, r in pd.read_csv(R / "E7_acoustic_vs_clock_differences.csv").iterrows():
        for m in ["acc", "bacc"]:
            add("E7 D2 acoustic vs clock", f"{r.population} | {r.comparison} | {m}", r[f"d_{m}"],
                r[f"d_{m}_lo"], r[f"d_{m}_hi"], "logger", r.n_clusters, kind="difference")

## 3. D3 (E5, E8)

In [4]:
if (R / "E5_recording_leakage_raw.csv").exists():
    e5 = pd.read_csv(R / "E5_recording_leakage_raw.csv")
    seed_ci(e5, ["protocol"], "E5 D3 v1 design")
    t5 = pd.DataFrame([c for c in cat if c["experiment"] == "E5 D3 v1 design"])
    t5.to_csv(R / "E5_recording_leakage_ci.csv", index=False)
if (R / "E5_oof_policies.csv").exists():
    for _, r in pd.read_csv(R / "E5_oof_policies.csv").iterrows():
        add("E5 D3 OOF", f"{r.decoding} | {r.policy} | acc", r.acc, r.run_boot_lo, r.run_boot_hi,
            "physical run", r.n_runs, note=f"init sd={r.init_sd:.4f}, split sd={r.split_sd:.4f}")
if (R / "E5_oof_policy_differences.csv").exists():
    for _, r in pd.read_csv(R / "E5_oof_policy_differences.csv").iterrows():
        for m in ["acc", "bacc", "excess"]:
            add("E5 D3 OOF", f"{r.step} | drop in {m}", r[f"d_{m}"], r[f"d_{m}_lo"], r[f"d_{m}_hi"],
                "physical run", r.n_clusters, kind="difference")
if (R / "E8_window_heuristic_raw.csv").exists():
    e8 = pd.read_csv(R / "E8_window_heuristic_raw.csv")
    seed_ci(e8, ["mode", "split"], "E8 D3 window heuristic")
if (R / "E8_minstd_vs_random_paired_runs.json").exists():
    r = la.load_json("E8_minstd_vs_random_paired_runs.json")
    add("E8 D3 window heuristic", "minstd - random (opcond LOCO)", r["diff"], r["lo"], r["hi"],
        "physical run", r["n_clusters"], kind="difference")

## 4. Catalogue

In [5]:
cat = pd.DataFrame(cat)
cat.to_csv(R / "intervals_catalogue.csv", index=False)
pd.set_option("display.max_colwidth", 70)
print(cat.groupby(["experiment", "unit", "kind"]).size().to_string())
print()
print(cat[cat.kind == "difference"].round(4).to_string(index=False))

experiment                    unit                                  kind      
E10 D1 mitigation             seed (paired split)                   difference     4
                              seed (split + initialisation)         level          6
E4 D1 collapse                seed (split + initialisation)         level         48
E4 D1 collapse (seed 0)       site                                  level         12
E4b D1 paired OOF             site                                  difference    27
E5 D3 OOF                     physical run                          difference    12
                                                                    level          5
E5 D3 v1 design               seed (split + initialisation)         level          3
E6 D2 clock stump             file (Wilson, files independent)      level          4
                              logger                                level          8
E7 D2 acoustic vs clock       logger                                dif